<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/dpo_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================================
# NB3-a · CELDA 1 — Setup, GPU y artefacto canónico compartido
# ============================================================================
import torch
assert torch.cuda.is_available(), "No hay GPU. Entorno de ejecución → Cambiar tipo → T4."
print("GPU:", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

# Unsloth fija TRL/transformers/peft compatibles. NO forzar versiones a mano.
!pip install -q unsloth          # descomentar solo en la 1ª corrida de la sesión

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/tesis_chatbot"         # <-- AJUSTAR a tu carpeta real
CONTEXTOS  = os.path.join(DRIVE_DIR, "contextos.json")
SET_GOLD   = os.path.join(DRIVE_DIR, "set_gold_FINAL.py")
SYS_TXT    = os.path.join(DRIVE_DIR, "system_canonico.txt")
CANDIDATOS = os.path.join(DRIVE_DIR, "candidatos.json")

# Artefacto canónico: system del agente de NB1 (celda 28), palabra por palabra.
# NB4 debe LEER este archivo, no redefinir el system → garantiza que Mistral se
# entrena bajo el mismo system con el que después se sirve.
SYSTEM_CANONICO = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si una FICHA advierte que referencia normas no vigentes, "
    "trasladá esa advertencia. Si el contexto no alcanza, decilo."
)
with open(SYS_TXT, "w", encoding="utf-8") as f:
    f.write(SYSTEM_CANONICO)
print("system_canonico.txt escrito.")

for p in (CONTEXTOS, SET_GOLD):
    print(("OK    " if os.path.exists(p) else "FALTA ") + p)

GPU: Tesla T4 | 15.6 GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================================
# NB3-a · CELDA 2 — Las 50 de BELEN con contexto top-2 de B_graphrag
# ============================================================================
# Prompt = join por ID: pregunta (set_gold) + top-2 de B_graphrag (contextos.json).
# EXCLUYE CASOS_TESTIGO (test set, §17.9c) y FASE2 (sin contexto aún, §17.11).
import json, importlib.util, statistics as st

with open(CONTEXTOS, encoding="utf-8") as f:
    CTX = json.load(f)["contextos"]

spec = importlib.util.spec_from_file_location("set_gold_FINAL", SET_GOLD)
sg = importlib.util.module_from_spec(spec); spec.loader.exec_module(sg)

TOP_K_DPO = 2   # §17.8: top-2 para el DPO (no 5)

def contexto_top2(qid):
    return "\n\n".join(CTX[qid]["B_graphrag"][:TOP_K_DPO])

def input_canonico(ctx, preg):     # idéntico a NB1 celda 28, línea 75
    return f"# Contexto:\n{ctx}\n\n# Pregunta:\n{preg}"

trabajo = []
for q in sg.PREGUNTAS_BELEN:
    qid = q["id"]
    if qid not in CTX:
        print("⚠ sin contexto:", qid); continue
    ctx = contexto_top2(qid)
    trabajo.append({"id": qid, "pregunta": q["pregunta"], "contexto": ctx,
                    "input": input_canonico(ctx, q["pregunta"])})

print(f"Preguntas de trabajo: {len(trabajo)}  (esperado 50)")
print("Contextos vacíos:", [t['id'] for t in trabajo if not t['contexto'].strip()] or "ninguno")

largos = sorted(trabajo, key=lambda t: len(t["input"]), reverse=True)
print("input chars → max:", len(largos[0]["input"]),
      "| mediana:", int(st.median([len(t["input"]) for t in trabajo])))
print("Top-3 más largos:", [(t["id"], len(t["input"])) for t in largos[:3]])

Preguntas de trabajo: 50  (esperado 50)
Contextos vacíos: ninguno
input chars → max: 6389 | mediana: 1961
Top-3 más largos: [('P2-08', 6389), ('P1-18', 5759), ('P1-21', 5331)]


In [5]:
# ============================================================================
# NB3-a · CELDA 3 — SMOKE-TEST de OOM
# ============================================================================
# Monta el DPOTrainer COMPLETO con max_seq_length=4096 sobre los inputs MÁS
# LARGOS de las 50 + respuestas dummy, corre 3 steps. Responde: ¿la T4 aguanta
# 4096 antes de generar un solo candidato? Si explota → bajar a 3072/2048.
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

MAX_SEQ, MAX_PROMPT = 4096, 3072          # deja ~1024 tokens para la respuesta

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=MAX_SEQ,
    dtype=None,                            # None → fp16 en T4 (no soporta bf16)
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

def to_prompt(inp):
    msgs = [{"role":"system","content":SYSTEM_CANONICO},
            {"role":"user","content":inp}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

relleno = "Según el contexto, la norma aplicable es la citada y su artículo. " * 60
dummy = [{"prompt": to_prompt(t["input"]), "chosen": relleno, "rejected": relleno[:200]}
         for t in largos[:4]]              # los 4 inputs más largos reales
ds = Dataset.from_list(dummy)

cfg = DPOConfig(
    output_dir="/content/smoke",
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    max_steps=3, learning_rate=5e-6, beta=0.1,
    max_length=MAX_SEQ, max_prompt_length=MAX_PROMPT,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    optim="adamw_8bit", logging_steps=1, report_to="none", seed=42,
)
trainer = DPOTrainer(
    model=model, ref_model=None,           # PEFT+Unsloth: referencia = adaptador off
    args=cfg, train_dataset=ds, processing_class=tokenizer,
)

torch.cuda.reset_peak_memory_stats()
trainer.train()
pico = torch.cuda.max_memory_reserved()/1e9
print(f"\n>>> SMOKE-TEST OK. VRAM pico: {pico:.1f}/15 GB. "
      f"{'Margen cómodo.' if pico < 13 else 'AL LÍMITE: bajar max_seq_length.'}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


AttributeError: '_OpNamespace' '_c10d_functional' object has no attribute '_wrap_tensor_autograd'

In [ ]:
# ============================================================================
# NB3-a · CELDA 4 — Generación de K=5 candidatos por pregunta (PILOTO)
# ============================================================================
# Genera con el instruct BASE (sin LoRA): los candidatos son las salidas que los
# abogados van a rankear; el DPO (NB3-b) viene DESPUÉS. §17.4.
#
# CLAVE (§17.3): el prompt que se GUARDA para el DPO es el canónico (system + input).
# La instrucción de estilo se usa SOLO para generar diversidad y NO se guarda —
# así los 5 candidatos comparten el mismo prompt y los pares son válidos.

import gc, torch
# Limpieza defensiva: descarta el modelo del smoke-test (LoRA dummy) si quedó cargado.
for v in ("trainer","model","tokenizer"):
    if v in dir():
        try: exec(f"del {v}")
        except: pass
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=4096, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)          # modo inferencia (2x más rápido)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

# --- Los 5 estilos (nombre, temperatura, instrucción de estilo) ---
ESTILOS = [
    {"nombre": "literal",     "temp": 0.0,
     "instr": "Respondé transcribiendo textualmente los artículos aplicables, sin interpretarlos ni simplificarlos."},
    {"nombre": "legalista",   "temp": 0.3,
     "instr": "Respondé con lenguaje jurídico formal y técnico."},
    {"nombre": "explicativa", "temp": 0.7,
     "instr": "Explicá de forma clara y accesible para un inversor sin formación jurídica, sin perder precisión."},
    {"nombre": "incompleta",  "temp": 0.7,
     "instr": "Respondé de forma breve, mencionando solo lo más básico."},
    {"nombre": "generica",    "temp": 1.0,
     "instr": "Respondé de forma general."},
]

def generar(input_canonico, instr, temp, max_new=512):
    # El prompt de GENERACIÓN lleva la instrucción de estilo en el turno del usuario;
    # el system se mantiene canónico e intacto.
    user = input_canonico + f"\n\n[Estilo de respuesta: {instr}]"
    msgs = [{"role": "system", "content": SYSTEM_CANONICO},
            {"role": "user",   "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    do_sample = temp > 0
    out = model.generate(
        **inputs, max_new_tokens=max_new,
        do_sample=do_sample,
        temperature=(temp if do_sample else None),
        top_p=(0.9 if do_sample else None),
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# --- PILOTO: 2 preguntas, los 5 estilos, para inspección a ojo ---
torch.manual_seed(42)
PILOTO = trabajo[:2]
for t in PILOTO:
    print("="*90)
    print(f"[{t['id']}] {t['pregunta']}")
    print("-"*90)
    for e in ESTILOS:
        txt = generar(t["input"], e["instr"], e["temp"])
        print(f"\n### {e['nombre'].upper()} (temp={e['temp']})\n{txt}")
    print()